# ENTERPRISE KNOWLEDGE MINING SYSTEM

#### DOCUMENT INGESTION

In [6]:
import urllib.parse
import urllib.request
import xml.etree.ElementTree as ET
import time

BASE_URL = "https://export.arxiv.org/api/query"

params = {
    "search_query": "all:computer",
    "start": 0,
    "max_results": 5
}

# Build query URL safely
query_url = BASE_URL + "?" + urllib.parse.urlencode(params)

request = urllib.request.Request(
    query_url,
    headers={
        "User-Agent": "arxiv-data-project/1.0 (research script)"
    }
)

time.sleep(3)

# Make request
with urllib.request.urlopen(request) as response:
    xml_data = response.read()

# Parse XML
root = ET.fromstring(xml_data)

# arXiv uses Atom namespace
ns = {
    "atom": "http://www.w3.org/2005/Atom"
}

papers = []

for entry in root.findall("atom:entry", ns):
    title = entry.find("atom:title", ns).text.strip()
    summary = entry.find("atom:summary", ns).text.strip()
    published = entry.find("atom:published", ns).text
    
    authors = [
        author.find("atom:name", ns).text
        for author in entry.findall("atom:author", ns)
    ]

    pdf_link = next((link.attrib.get("href") for link in entry.findall("atom:link", ns) 
                if link.attrib.get("title") == "pdf" and link.attrib.get("type") == "application/pdf" and link.attrib.get("rel") == "related"), 
                None
            )

    papers.append({
        "title": title,
        "authors": authors,
        "published": published,
        "summary": summary,
        "pdf_link": pdf_link
    })

# Display parsed results
for paper in papers:
    print("\nTITLE:", paper["title"])
    print("AUTHORS:", ", ".join(paper["authors"]))
    print("PUBLISHED:", paper["published"])
    print("SUMMARY:", paper["summary"][:200], "...")
    print("PDF LINK:", paper["pdf_link"])


TITLE: Vision Based Game Development Using Human Computer Interaction
AUTHORS: S. Sumathi, S. K. Srivatsa, M. Uma Maheswari
PUBLISHED: 2010-02-10T19:46:07Z
SUMMARY: A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines. The system presented here is a vision-based system for detection of long vo ...
PDF LINK: https://arxiv.org/pdf/1002.2191v1

TITLE: From truth to computability I
AUTHORS: Giorgi Japaridze
PUBLISHED: 2004-07-21T03:58:22Z
SUMMARY: The recently initiated approach called computability logic is a formal theory of interactive computation. See a comprehensive online source on the subject at http://www.cis.upenn.edu/~giorgi/cl.html . ...
PDF LINK: https://arxiv.org/pdf/cs/0407054v2

TITLE: Changing Neighbors k Secure Sum Protocol for Secure Multi Party Computation
AUTHORS: Rashid Sheikh, Beerendra Kumar, Durgesh Kumar Mishra
PUBLISHED: 2010-02-11T19:58:10Z
SUMMARY: Secure sum computation of private data inpu

#### EXTRACTING PDF AND CONVERTING TO MARKDOWN

In [7]:
# trying pymupdf4llm
import pymupdf4llm
import urllib.request
import os

pdf_dir = "pdfs"
os.makedirs(pdf_dir, exist_ok = True)

pdf_path = "https://arxiv.org/pdf/1002.2191v1"
local_pdf = os.path.join(pdf_dir, "Vision Based Game Development Using HCI.pdf")
urllib.request.urlretrieve(pdf_path, local_pdf)

pages = pymupdf4llm.to_markdown(local_pdf, page_chunks=True)

print(f"Pages extracted: {len(pages)}")
print(pages[0]["metadata"])
print(pages[0]["text"][:1000])


OCR disabled because Tesseract language data not found.
Pages extracted: 7
{'format': 'PDF 1.4', 'title': 'Microsoft Word - 07011060 IJCSIS Camera Ready Paper.doc', 'author': 'RAZVI', 'subject': '', 'keywords': '', 'creator': 'PScript5.dll Version 5.2.2', 'producer': 'Acrobat Distiller 8.1.0 (Windows)', 'creationDate': "D:20100202231332+05'00'", 'modDate': "D:20100208131620+05'00'", 'trapped': '', 'encryption': None, 'file_path': 'pdfs\\Vision Based Game Development Using HCI.pdf', 'page_count': 7, 'page_number': 1}
_(IJCSIS) International Journal of Computer Science and Information Security, Vol. 7, No. 1, 2010_ 

## Vision Based Game Development Using Human Computer Interaction 

Bharath  University                   St.Joseph College of Engineering                   Bharath University Chennai,India                                       Chennai,India                                     Chennai,India 

_**Abstract**_ **— A Human Computer Interface (HCI) System for playing games is des

In [8]:
#checking for short/empty pages that might cause issues with NER
for page in pages:
    if len(page["text"].strip()) <50:
         print(f"Short/empty page: {page['metadata']}")

#checking the character count distribution
lengths = [len(p["text"]) for p in pages]
print(f'Average chars per page: {sum(lengths)/len(lengths):.0f}')
print(f"Min: {min(lengths)}, Max: {max(lengths)}")

Average chars per page: 3487
Min: 2493, Max: 4217


#### DATA CLEANING FOR CHUNKING

In [66]:
import re

def clean_text(text: str) -> str:
    if not text:
        return ""

    # Normalize line endings
    text = text.replace("\r\n", "\n").replace("\r", "\n")

    # Remove markdown heading markers and separator-only lines
    text = re.sub(r"(?m)^\s*#{1,6}\s*", " ", text)
    text = re.sub(r"(?m)^\s*[-*_~=`]{2,}\s*$", " ", text)

    # Remove markdown emphasis symbols anywhere in text
    text = re.sub(r"[_*`~]+", " ", text)

    # Remove hash symbols that can confuse NER
    text = re.sub(r"(?<!\w)#(?=\w)", "", text)
    text = text.replace("#", " ")

    # Fix hyphenated line breaks: learn-\ning -> learning
    text = re.sub(r"([A-Za-z])-\s*\n\s*([A-Za-z])", r"\1\2", text)

    # Remove obvious page-only lines
    text = re.sub(r"(?im)^\s*page\s*\d+(\s*of\s*\d+)?\s*$", " ", text)
    text = re.sub(r"(?m)^\s*\d+\s*$", " ", text)

    # Normalize long dashes
    text = re.sub(r"[—–]+", " - ", text)

    # Remove leading punctuation/whitespace 
    text = re.sub(r"^[\s\.,;:-]+", "", text)

    # Collapse repeated whitespace/newlines
    text = re.sub(r"\n{2,}", "\n", text)
    text = text.replace("\n", " ")

    # Final whitespace cleanup
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [67]:
def clean_page_text(text: str, page_number=None) -> str:
    text = clean_text(text)

    if not text:
        return ""

    # First-page specific cleanup:
    # if Abstract exists, keep text starting from Abstract
    if page_number == 1:
        match = re.search(r"\babstract\b\s*[:-]?\s*", text, flags=re.IGNORECASE)
        if match:
            text = text[match.start():]

    return re.sub(r"\s+", " ", text).strip()

In [68]:
clean_pages = []
for page in pages:
    cleaned = clean_page_text(
        page["text"],
        page_number=page["metadata"].get("page_number")
    )
    clean_pages.append({
        "text": cleaned,
        "metadata": page["metadata"]
    })

print("Pages before:", len(pages))
print("Pages after cleaning:", len(clean_pages))
print("Sample cleaned page text:", clean_pages[0]["text"][:500] if clean_pages else "N/A")

Pages before: 7
Pages after cleaning: 7
Sample cleaned page text: Abstract - A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines. The system presented here is a vision-based system for detection of long voluntary eye blinks and interpretation of blink patterns for communication between man and machine. This system replaces the mouse with the human face as a new way to interact with the computer. Facial features (nose tip and eyes) are detected and tracked in real-time to use their actions 


In [78]:
#chunking the cleaned text for spacy using langchain text splitters
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)

paper = papers[0] if papers else {}

paper_id = paper.get("id", "paper_0")
title = paper.get("title", "")
authors = ", ".join(paper.get("authors", [])) if paper.get("authors") else ""
published = paper.get("published", "")
abstract = paper.get("summary", "")

chunks = []

for page in clean_pages:
    splits = text_splitter.split_text(page["text"])

    for s in splits:
        s = re.sub(r"^[\s\.,;:-]+", "", s) #remove any leading punctuation

        if s.strip():  # skip empty chunks
            chunks.append({
                "text": s,
                "metadata": {
                    "paper_id": paper_id,
                    "title": title,
                    "authors": authors,
                    "published": published,
                    "creation_date": page["metadata"].get("creationDate", ""),
                    "page_number": page["metadata"].get("page_number", ""),
                    "page_count": page["metadata"].get("page_count", ""),
                    "format": page["metadata"].get("format", "")
                }
            })

# Add chunk index after creation
for i, chunk in enumerate(chunks):
    chunk["metadata"]["chunk_index"] = i

print("Total chunks:", len(chunks))
print(chunks[0] if chunks else "No chunks created")

Total chunks: 75
{'text': 'Abstract - A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines. The system presented here is a vision-based system for detection of long voluntary eye blinks and interpretation of blink patterns for communication between man and machine. This system replaces the mouse with the human face as a new way to interact with the computer', 'metadata': {'paper_id': 'paper_0', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'authors': 'S. Sumathi, S. K. Srivatsa, M. Uma Maheswari', 'published': '2010-02-10T19:46:07Z', 'creation_date': "D:20100202231332+05'00'", 'page_number': 1, 'page_count': 7, 'format': 'PDF 1.4', 'chunk_index': 0}}


In [79]:
# separating metadata from text for NER to avoid noise
def text_for_ner(chunk):
    text = (chunk.get("text") or "")
    md = chunk.get("metadata", {})

    # Clean special chars that confuse NER
    text = re.sub(r"[*_`~^—–]+", " ", text)   # removed plain hyphen here
    
    #Remove leading punctuation/whitespace that can confuse NER
    text = re.sub(r"^[\s\.,;:-]+", "", text)

    if md.get("page_number") == 1:
        title = (md.get("title") or "")
        if title:
            text = re.sub(re.escape(title), " ", text, flags=re.IGNORECASE)

        authors = md.get("authors", "")
        if isinstance(authors, str):
            author_list = [a.strip() for a in authors.split(",") if a.strip()]
        elif isinstance(authors, list):
            author_list = [str(a).strip() for a in authors if str(a).strip()]
        else:
            author_list = []

        for author in author_list:
            text = re.sub(rf"\b{re.escape(author)}\b", " ", text, flags=re.IGNORECASE)

        # Keep only text after Abstract
        parts = re.split(r"\babstract\b[:\-\s]*", text, maxsplit=1, flags=re.IGNORECASE)
        if len(parts) == 2:
            text = parts[1]

    return re.sub(r"\s+", " ", text).strip()

#### Entity Extraction

In [80]:
#using spacy model for NER

import spacy

nlp = spacy.load("en_core_web_md") # using at the moment since it has better NER performance than the small mode
# nlp = spacy.load("en_core_web_sm") # would eventually switch when increase research papers for faster performance

In [82]:
def clean_entities(raw_entities):
    cleaned = []
    seen = set()

    for ent_text, ent_label in raw_entities:
        ent_text = ent_text.strip()

        # skip empty entities
        if not ent_text:
            continue

        # remove very short fragments like "I."
        if len(ent_text) < 3:
            continue

        # remove pure numbers
        if re.fullmatch(r"\d+(\.\d+)?", ent_text):
            continue

        # skip ordinal/cardinal noise
        if ent_label in {"CARDINAL", "ORDINAL"}:
            continue

        # remove mostly punctuation fragments
        if re.fullmatch(r"[^\w]+", ent_text):
            continue

        key = (ent_text.lower(), ent_label)
        if key not in seen:
            seen.add(key)
            cleaned.append((ent_text, ent_label))

    return cleaned

In [83]:
# updating metadata with NER results 
ner_texts = [text_for_ner(c) for c in chunks]
docs = nlp.pipe(ner_texts, batch_size=32)

for i, doc in enumerate(docs):
    raw_entities = [(ent.text, ent.label_) for ent in doc.ents]
    cleaned_entities = clean_entities(raw_entities)

    chunks[i]["metadata"]["entities_raw"] = raw_entities
    chunks[i]["metadata"]["entities_clean"] = cleaned_entities
    chunks[i]["metadata"]["entities"] = ", ".join([ent_text for ent_text, _ in cleaned_entities])
    chunks[i]["metadata"]["entity_labels"] = ", ".join([ent_label for _, ent_label in cleaned_entities])
    chunks[i]["metadata"]["ner_text"] = ner_texts[i]

    if i < 5:
        print("\nChunk", i, "cleaned entities:", cleaned_entities[:20])
        print("Chunk", i, "NER text preview:", ner_texts[i][:180])


Chunk 0 cleaned entities: [('Human Computer Interface', 'ORG')]
Chunk 0 NER text preview: A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines. The system presented here is a vision-based system for 

Chunk 1 cleaned entities: []
Chunk 1 NER text preview: This system replaces the mouse with the human face as a new way to interact with the computer. Facial features (nose tip and eyes) are detected and tracked in real-time to use thei

Chunk 2 cleaned entities: [('USB', 'ORG'), ('Human Computer Interface', 'ORG'), ('HCI', 'ORG'), ('SSR Filter', 'ORG'), ('Hough', 'PERSON')]
Chunk 2 NER text preview: The left/right eye blinks fire left/right mouse click events. The system works with inexpensive USB cameras and runs at a frame rate of 30 frames per second. Keywords: Human Comput

Chunk 3 cleaned entities: [('Human-Computer Interface', 'ORG'), ('HCI', 'ORG')]
Chunk 3 NER text preview: Human-Computer Interface (HCI) can b

In [84]:
# printing the metadata of the first chunk to verify NER results are stored
print(chunks[0]["metadata"])

{'paper_id': 'paper_0', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'authors': 'S. Sumathi, S. K. Srivatsa, M. Uma Maheswari', 'published': '2010-02-10T19:46:07Z', 'creation_date': "D:20100202231332+05'00'", 'page_number': 1, 'page_count': 7, 'format': 'PDF 1.4', 'chunk_index': 0, 'entities_raw': [('Human Computer Interface', 'ORG')], 'entities_clean': [('Human Computer Interface', 'ORG')], 'entities': 'Human Computer Interface', 'entity_labels': 'ORG', 'ner_text': 'A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines. The system presented here is a vision-based system for detection of long voluntary eye blinks and interpretation of blink patterns for communication between man and machine. This system replaces the mouse with the human face as a new way to interact with the computer'}


In [85]:
final_chunks = []

for i, chunk in enumerate(chunks):
    final_chunks.append({
        "text": chunk["text"],
        "metadata": {
            "paper_id": chunk["metadata"].get("paper_id", "paper_0"),
            "title": chunk["metadata"].get("title", ""),
            "authors": chunk["metadata"].get("authors", ""),
            "published": chunk["metadata"].get("published", ""),
            "creation_date": chunk["metadata"].get("creation_date", ""),
            "page_number": chunk["metadata"].get("page_number", ""),
            "page_count": chunk["metadata"].get("page_count", ""),
            "format": chunk["metadata"].get("format", ""),
            "chunk_index": i,
            "entities": chunk["metadata"].get("entities", ""),
            "entity_labels": chunk["metadata"].get("entity_labels", "")
        }
    })

    if i < 5:
        print("\nFinal Chunk", i, "metadata:", final_chunks[-1]["metadata"])
        print("Final Chunk", i, "text preview:", final_chunks[-1]["text"][:180])


Final Chunk 0 metadata: {'paper_id': 'paper_0', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'authors': 'S. Sumathi, S. K. Srivatsa, M. Uma Maheswari', 'published': '2010-02-10T19:46:07Z', 'creation_date': "D:20100202231332+05'00'", 'page_number': 1, 'page_count': 7, 'format': 'PDF 1.4', 'chunk_index': 0, 'entities': 'Human Computer Interface', 'entity_labels': 'ORG'}
Final Chunk 0 text preview: Abstract - A Human Computer Interface (HCI) System for playing games is designed here for more natural communication with the machines. The system presented here is a vision-based 

Final Chunk 1 metadata: {'paper_id': 'paper_0', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'authors': 'S. Sumathi, S. K. Srivatsa, M. Uma Maheswari', 'published': '2010-02-10T19:46:07Z', 'creation_date': "D:20100202231332+05'00'", 'page_number': 1, 'page_count': 7, 'format': 'PDF 1.4', 'chunk_index': 1, 'entities': '', 'entity_labels': ''}
Final Chunk 1 

#### EMBEDDING WITH OPENAI aAND STORING IN CHROMADB

In [ ]:
from openai import OpenAI
import os
import dotenv
import chromadb

dotenv.load_dotenv()  # Load environment variables from .env file

client = OpenAI(api_key=os.getenv("OPENAI_KEY"))
embedding_model = "text-embedding-3-small" # will change to text-embedding-3-large when increase research papers for better performance

# using persistent chroma db to store the embeddings and metadata for later retrieval and analysis
chroma_client = chromadb.PersistentClient(path="./chroma_db")

collection_name = "research_papers"

try:
    chroma_client.delete_collection(collection_name)
    print(f"Deleted existing collection: {collection_name}")
except:
    print(f"No existing collection named: {collection_name}")

collection = chroma_client.get_or_create_collection(name=collection_name)
print("Chroma collection ready.")

Deleted existing collection: research_papers
Chroma collection ready.


In [93]:
#preparing data for insertion into chroma db
documents = [chunk["text"] for chunk in final_chunks]
metadatas = [chunk["metadata"] for chunk in final_chunks]
ids = [
    f"{chunk['metadata']['paper_id']}_chunk_{chunk['metadata']['chunk_index']}"
    for chunk in final_chunks
]

print("Documents:", len(documents))
print("Metadatas:", len(metadatas))
print("IDs:", len(ids))
print("Sample ID:", ids[0] if ids else "N/A")
print("Sample metadata:", metadatas[0] if metadatas else "N/A")

Documents: 75
Metadatas: 75
IDs: 75
Sample ID: paper_0_chunk_0
Sample metadata: {'paper_id': 'paper_0', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'authors': 'S. Sumathi, S. K. Srivatsa, M. Uma Maheswari', 'published': '2010-02-10T19:46:07Z', 'creation_date': "D:20100202231332+05'00'", 'page_number': 1, 'page_count': 7, 'format': 'PDF 1.4', 'chunk_index': 0, 'entities': 'Human Computer Interface', 'entity_labels': 'ORG'}


In [94]:
#batching embedding
def get_embeddings(texts, model="text-embedding-3-small"):
    response = client.embeddings.create(
        model=model,
        input=texts
    )
    return [item.embedding for item in response.data]

In [95]:
batch_size = 20
all_embeddings = []

for i in range(0, len(documents), batch_size):
    batch_docs = documents[i:i + batch_size]
    batch_embeddings = get_embeddings(batch_docs, model=embedding_model)
    all_embeddings.extend(batch_embeddings)
    print(f"Embedded chunks {i} to {i + len(batch_docs) - 1}")

print("Total embeddings created:", len(all_embeddings))
print("Embedding dimension:", len(all_embeddings[0]) if all_embeddings else 0)

Embedded chunks 0 to 19
Embedded chunks 20 to 39
Embedded chunks 40 to 59
Embedded chunks 60 to 74
Total embeddings created: 75
Embedding dimension: 1536


In [ ]:
#storing embeddings and metadata in chroma db
collection.add(
    ids=ids,
    documents=documents,
    metadatas=metadatas,
    embeddings=all_embeddings
)

print(f"Stored {len(documents)} chunks in Chroma collection '{collection_name}'.")

Stored 75 chunks in Chroma collection 'research_papers'.


### Testing Retrieval

In [113]:
def retrieve_chunks(query, n_results=3):
    query_embedding = get_embeddings([query], model=embedding_model)[0]
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=n_results
    )
    return results

In [114]:
test_queries = [
    "What is the main purpose of the proposed HCI system?",
    "What is Hough transform and how is it used in this paper?",
    "How does blink detection work in the eye ROI?"
]

for query in test_queries:
    print("\n" + "="*50)
    print(f"\nSearch results for query: '{query}'")
    results = retrieve_chunks(query, n_results=3)

    for i in range(len(results["documents"][0])):
        print(f"\nResult {i+1}")
        print("ID:", results["ids"][0][i])
        print("Metadata:", results["metadatas"][0][i])
        print("Text:", results["documents"][0][i][:400])
    print("\n" + "="*50)



Search results for query: 'What is the main purpose of the proposed HCI system?'

Result 1
ID: paper_0_chunk_3
Metadata: {'paper_id': 'paper_0', 'chunk_index': 3, 'authors': 'S. Sumathi, S. K. Srivatsa, M. Uma Maheswari', 'title': 'Vision Based Game Development Using Human Computer Interaction', 'page_number': 1, 'published': '2010-02-10T19:46:07Z', 'format': 'PDF 1.4', 'page_count': 7, 'entities': 'Human-Computer Interface, HCI', 'creation_date': "D:20100202231332+05'00'", 'entity_labels': 'ORG, ORG'}
Text: Human-Computer Interface (HCI) can be described as the point of communication between the human and a computer. HCI aims to use human features to interact with the computer. The system tracks the computer user's movements with a video camera and translates them into the movements of the mouse pointer on the screen. The tip of the user's nose can be tracked and captured with a webcam and monitor it

Result 2
ID: paper_0_chunk_0
Metadata: {'entities': 'Human Computer Interface', 'p

### RAG PIPELINE IMPLEMENTATION

In [137]:
RAG_SYSTEM_PROMPT = """
You are an academic research assistant for an enterprise knowledge mining system built on arXiv papers.

Instructions:
- Answer questions only from the retrieved context.
- Treat the retrieved text as excerpts from research papers.
- Prefer precise academic wording.
- Summarize contributions, methods, datasets, results, or limitations only if they are supported by the context.
- If the answer is incomplete or missing from the context, say that explicitly, eg: "I cannot answer this question based on the provided context. I need more clarification or additional sources to answer this question."
- Do not hallucinate citations or details.
- Do not include source tags or citation brackets such as [Source 1] in the answer text.
- Write only the answer and ensure it is formatted concisely and clearly.
"""

In [138]:
def generate_answer(query, context, max_tokens=300, temperature=0.2):
    response = client.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": RAG_SYSTEM_PROMPT},
            {
                "role": "user",
                "content": (
                    f"Use the retrieved research-paper context below to answer the question.\n\n"
                    f"Context:\n{context}\n\n"
                    f"Question: {query}\n\n"
                )
            }
        ],
        max_tokens=max_tokens,
        temperature=temperature
    )

    return response.choices[0].message.content.strip()

In [139]:
def rag_query(query, n_results=3):
    results = retrieve_chunks(query, n_results=n_results)

    context = ""
    sources = []

    for i in range(len(results["documents"][0])):
        chunk_text = results["documents"][0][i]
        chunk_metadata = results["metadatas"][0][i]
        source_id = results["ids"][0][i]

        context += (
            f"[Source {i+1}: {chunk_metadata.get('title', 'Unknown Title')}, "
            f"Page {chunk_metadata.get('page_number', '?')}, "
            f"Chunk {chunk_metadata.get('chunk_index', '?')}]\n"
            f"{chunk_text}\n\n"
        )

        sources.append({
            "source_number": i + 1,
            # "chunk_id": source_id,
            "title": chunk_metadata.get("title", ""),
            "page_number": chunk_metadata.get("page_number", ""),
            # "chunk_index": chunk_metadata.get("chunk_index", "")
        })

    answer = generate_answer(query, context)

    return {
        "query": query,
        "answer": answer,
        "sources": sources
    }

In [141]:
queries=[
    "What is the main purpose of the proposed HCI system?",
    "How does the system use the nose tip and eye blinks to replace mouse actions?",
    "What is the role of the SSR filter in the face detection process?",
    "What is Hough transform and how is it used in this paper?",
    "How does the system detect eye blinks and motion in the eye region?"
]

for q in queries:
    print("\n" + "="*50)
    print(f"\nRAG response for query: '{q}'")
    response = rag_query(q, n_results=3)
    print("Answer:\n", response["answer"])
    print("\nSources:")
    for s in response["sources"]:
        print(s)



RAG response for query: 'What is the main purpose of the proposed HCI system?'
Answer:
 The main purpose of the proposed Human-Computer Interface (HCI) system is to enable more natural communication between humans and machines by using vision-based detection of long voluntary eye blinks and blink patterns to replace traditional mouse input, allowing the human face to serve as a new mode of interaction with the computer.

Sources:
{'source_number': 1, 'title': 'Vision Based Game Development Using Human Computer Interaction', 'page_number': 1}
{'source_number': 2, 'title': 'Vision Based Game Development Using Human Computer Interaction', 'page_number': 1}
{'source_number': 3, 'title': 'Vision Based Game Development Using Human Computer Interaction', 'page_number': 1}


RAG response for query: 'How does the system use the nose tip and eye blinks to replace mouse actions?'
Answer:
 The system detects and tracks facial features in real-time, specifically the nose tip and eyes. The coordin